In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import os
import sys

In [0]:
current_dir = os.getcwd()
sys.path.append(current_dir)

# CUSTOMERS

In [0]:
cust_df = spark.read.table("pysparkdbt.bronze.customers")

cust_df = cust_df.withColumn("domain",split('email','@')[1])\
                .withColumn("phone_number",regexp_replace('phone_number',r"[^0-9]",""))\
                .withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))\
                .drop("first_name","last_name")
cust_df.display()

In [0]:
from utils.custom_utils import transformation

cust_obj = transformation()
cust_df_trns = cust_obj.dedup(cust_df,['customer_id'],'last_updated_timestamp')
cust_df_trns = cust_obj.process_timestamp(cust_df_trns)
cust_df_trns.display()


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.customers"):
  cust_df_trns.write.format("delta").mode("append").saveAsTable("pysparkdbt.silver.customers")
else:
  cust_obj.upsert(cust_df_trns,['customer_id'],'customers','last_updated_timestamp')

In [0]:
%sql
SELECT count(*) FROM pysparkdbt.silver.customers

# DRIVERS

In [0]:
driver_df = spark.read.table("pysparkdbt.bronze.drivers")

driver_df = driver_df.withColumn("phone_number",regexp_replace('phone_number',r"[^0-9]",""))\
                .withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))\
                .drop("first_name","last_name")
driver_df.display()

In [0]:
driver_obj = transformation()
driver_df_trns = driver_obj.dedup(driver_df,['driver_id'],'last_updated_timestamp')
driver_df_trns = driver_obj.process_timestamp(driver_df_trns)
driver_df_trns.display()

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):
  driver_df_trns.write.format("delta").mode("append").saveAsTable("pysparkdbt.silver.drivers")
else:
  driver_obj.upsert(driver_df_trns,['driver_id'],'drivers','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.drivers

# Locations

In [0]:
loc_df = spark.read.table("pysparkdbt.bronze.locations")
loc_df.display()

In [0]:
loc_obj = transformation()
loc_df_trns = loc_obj.dedup(loc_df,['location_id'],'last_updated_timestamp')
loc_df_trns = loc_obj.process_timestamp(loc_df_trns)
loc_df_trns.display()

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.locations"):
  loc_df_trns.write.format("delta").mode("append").saveAsTable("pysparkdbt.silver.locations")
else:
  loc_obj.upsert(loc_df_trns,['location_id'],'locations','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.locations

# Payments

In [0]:
pay_df = spark.read.table("pysparkdbt.bronze.payments")
pay_df = pay_df.withColumn('online_payment_status',
                           when(((col('payment_method')=='Card') & (col('payment_status')=='Success')),"Online-success")
                           .when(((col('payment_method')=='Card') & (col('payment_status')=='Failed')),"Online-Failed")
                           .when(((col('payment_method')=='Card') & (col('payment_status')=='Pending')),"Online-Pending")
                           .otherwise("offline"))
pay_df.display()

In [0]:
payment_obj = transformation()
pay_df_trns = payment_obj.dedup(pay_df,['payment_id'],'last_updated_timestamp')
pay_df_trns = payment_obj.process_timestamp(pay_df_trns)
pay_df_trns.display()

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.payments"):
  pay_df_trns.write.format("delta").mode("append").saveAsTable("pysparkdbt.silver.payments")
else:
  payment_obj.upsert(pay_df_trns,['payment_id'],'payments','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.payments

# Vehicles

In [0]:
veh_df = spark.read.table("pysparkdbt.bronze.vehicles")
veh_df = veh_df.withColumn('make',upper(col('make')))
veh_df.display()

In [0]:
veh_obj = transformation()
veh_df_trns = veh_obj.dedup(veh_df,['vehicle_id'],'last_updated_timestamp')
veh_df_trns = veh_obj.process_timestamp(veh_df_trns)
veh_df_trns.display()


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):
  veh_df_trns.write.format("delta").mode("overwrite").saveAsTable("pysparkdbt.silver.vehicles")
else:
  veh_obj.upsert(veh_df_trns,['vehicle_id'],'vehicles','last_updated_timestamp')

In [0]:
%sql
SELECT COUNT(*) FROM pysparkdbt.silver.vehicles